# Telco Customer Churn — Model Training

## Objective

Train and compare machine learning models for customer churn prediction.

### Models

- Logistic Regression
- Random Forest

### Evaluation

The models will be compared using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("Evaluation metrics imported successfully!")

Evaluation metrics imported successfully!


In [3]:
print("Model training notebook is ready!")

Model training notebook is ready!


In [4]:
from sklearn.pipeline import Pipeline

print("Pipeline imported successfully!")

Pipeline imported successfully!


In [6]:
from pathlib import Path

project_root = Path("..")

csv_files = list(project_root.rglob("*.csv"))

print("CSV files found:")
for file in csv_files:
    print(file)

CSV files found:
..\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
..\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
..\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-2.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-2.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-2.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\philox-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\philox-testset-2.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\sfc64-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\sfc64-testset-2.csv
..\.venv\Lib\site-packages\numpy\_core\tests\

In [7]:
from sklearn.pipeline import Pipeline

print("Pipeline imported successfully!")

Pipeline imported successfully!


In [8]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Logistic Regression pipeline created successfully!")

NameError: name 'preprocessor' is not defined

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessor created successfully!")

NameError: name 'X_train' is not defined

In [10]:
from pathlib import Path

# Find the dataset
project_root = Path("..")
csv_files = list(project_root.rglob("*.csv"))

print("CSV files found:")
for file in csv_files:
    print(file)

CSV files found:
..\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
..\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
..\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-2.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-2.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-2.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\philox-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\philox-testset-2.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\sfc64-testset-1.csv
..\.venv\Lib\site-packages\numpy\random\tests\data\sfc64-testset-2.csv
..\.venv\Lib\site-packages\numpy\_core\tests\

In [11]:
data_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (7043, 21)


In [12]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Create TotalServices
service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["TotalServices"] = (
    df[service_columns]
    .replace({
        "Yes": 1,
        "No": 0,
        "No phone service": 0,
        "No internet service": 0
    })
    .astype(int)
    .sum(axis=1)
)

# Create TenureGroup
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, np.inf],
    labels=[
        "0-12 months",
        "13-24 months",
        "25-48 months",
        "49+ months"
    ]
)

# Create AverageMonthlySpend
df["AverageMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

# Encode target
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Remove identifier
df = df.drop(columns=["customerID"])

print("Feature preparation completed!")
print("Shape:", df.shape)

Feature preparation completed!
Shape: (7043, 23)


In [13]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (5634, 22)
Testing set: (1409, 22)


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


C:\Users\hp\AppData\Local\Temp\ipykernel_15424\783686037.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [15]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Logistic Regression pipeline created successfully!")

Logistic Regression pipeline created successfully!


In [16]:
logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [17]:
from sklearn.pipeline import Pipeline

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Logistic Regression pipeline created successfully!")

Logistic Regression pipeline created successfully!


In [18]:
logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [19]:
y_pred_logistic = logistic_pipeline.predict(X_test)

y_proba_logistic = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")

Predictions generated successfully!


In [20]:
logistic_accuracy = accuracy_score(y_test, y_pred_logistic)
logistic_precision = precision_score(y_test, y_pred_logistic)
logistic_recall = recall_score(y_test, y_pred_logistic)
logistic_f1 = f1_score(y_test, y_pred_logistic)
logistic_roc_auc = roc_auc_score(y_test, y_proba_logistic)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy : {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall   : {logistic_recall:.4f}")
print(f"F1-Score : {logistic_f1:.4f}")
print(f"ROC-AUC  : {logistic_roc_auc:.4f}")

Logistic Regression Results
---------------------------
Accuracy : 0.7991
Precision: 0.6532
Recall   : 0.5187
F1-Score : 0.5782
ROC-AUC  : 0.8421


In [21]:
print(
    classification_report(
        y_test,
        y_pred_logistic,
        target_names=["No Churn", "Churn"]
    )
)

              precision    recall  f1-score   support

    No Churn       0.84      0.90      0.87      1035
       Churn       0.65      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [22]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Random Forest pipeline created successfully!")

Random Forest pipeline created successfully!


In [23]:
random_forest_pipeline.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [24]:
y_pred_rf = random_forest_pipeline.predict(X_test)

y_proba_rf = random_forest_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest predictions generated successfully!")

Random Forest predictions generated successfully!


In [25]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_roc_auc = roc_auc_score(y_test, y_proba_rf)

print("Random Forest Results")
print("---------------------")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")

Random Forest Results
---------------------
Accuracy : 0.7764
Precision: 0.5987
Recall   : 0.4786
F1-Score : 0.5319
ROC-AUC  : 0.8208


In [26]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        logistic_accuracy,
        rf_accuracy
    ],
    "Precision": [
        logistic_precision,
        rf_precision
    ],
    "Recall": [
        logistic_recall,
        rf_recall
    ],
    "F1-Score": [
        logistic_f1,
        rf_f1
    ],
    "ROC-AUC": [
        logistic_roc_auc,
        rf_roc_auc
    ]
})

results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.799148,0.653199,0.518717,0.578241,0.842148
1,Random Forest,0.776437,0.598662,0.478610,0.531947,0.820810


In [27]:
cm_logistic = confusion_matrix(
    y_test,
    y_pred_logistic
)

print("Logistic Regression Confusion Matrix:")
print(cm_logistic)

Logistic Regression Confusion Matrix:
[[932 103]
 [180 194]]


In [28]:
cm_rf = confusion_matrix(
    y_test,
    y_pred_rf
)

print("Random Forest Confusion Matrix:")
print(cm_rf)

Random Forest Confusion Matrix:
[[915 120]
 [195 179]]


In [29]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

print("Processed directory ready!")

Processed directory ready!


In [30]:
results.to_csv(
    processed_dir / "model_comparison.csv",
    index=False
)

print("Model comparison saved successfully!")

Model comparison saved successfully!


## Model Training Summary

Two machine learning models were trained and evaluated:

1. Logistic Regression
2. Random Forest

The models were evaluated using:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
- Confusion Matrix

The best-performing model will be selected based on the evaluation results.

The next stage will focus on deeper model evaluation and Explainable AI using SHAP.